# Adapter / MID trimming — cutadapt only

Human `PRJEB40348` trim notebook. Removes technical A-key, B-key+MID prefixes, and Illumina platform adapter. Does **not** trim VH/JH primers. Writes `results/<dataset>/trimmed/fastq`. No QC is generated here.

In [ ]:
import os, sys, sysconfig, shutil, subprocess
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [_CONDA_ENV + "/lib/python3.11/site-packages", sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)


In [ ]:
from pathlib import Path

TRIM_QUALITY = '0,30'
MIN_LENGTH = 250
ADAPTER_TIMES = 2
A_KEY = 'CGTATCGCCTCCCTCGCGCCATCAGACGCCTCGAGGCGGCCGCTCTAGA'
B_KEYS = ['CTATGCGCCTTGCCAGCCCGCTCAGACGAGTGCGTACTTGGCTAGCGCCAAGCTTGCTGA', 'CTATGCGCCTTGCCAGCCCGCTCAGACGCTCGACAACTTGGCTAGCGCCAAGCTTGCTGA', 'CTATGCGCCTTGCCAGCCCGCTCAGAGACGCACTCACTTGGCTAGCGCCAAGCTTGCTGA', 'CTATGCGCCTTGCCAGCCCGCTCAGAGCACTGTAGACTTGGCTAGCGCCAAGCTTGCTGA', 'CTATGCGCCTTGCCAGCCCGCTCAGATCAGACACGACTTGGCTAGCGCCAAGCTTGCTGA', 'CTATGCGCCTTGCCAGCCCGCTCAGATATCGCGAGACTTGGCTAGCGCCAAGCTTGCTGA']
PLATFORM_ADAPTER = 'AGATCGGAAGAGCGGTTCAG'


In [ ]:
def run_adapter_trim(volume, dataset):
    vol = Path(volume)
    src = vol / 'raw' / dataset
    base = vol / 'results' / dataset / 'trimmed'
    out = base / 'fastq'

    if base.exists():
        shutil.rmtree(base)

    out.mkdir(parents=True, exist_ok=True)

    pairs = sorted(set(f.name.rsplit('_', 1)[0] for f in src.glob('*.fastq.gz')))
    print(f'[adapter_trim] {dataset}: {len(pairs)} pairs')
    print(f'[adapter_trim] quality={TRIM_QUALITY} minlen={MIN_LENGTH} times={ADAPTER_TIMES}')

    for bn in pairs:
        r1 = src / f'{bn}_1.fastq.gz'
        r2 = src / f'{bn}_2.fastq.gz'
        if not r1.exists() or not r2.exists():
            continue

        o1 = out / f'{bn}_1.trim.fastq.gz'
        o2 = out / f'{bn}_2.trim.fastq.gz'

        ca = ['cutadapt', '--quality-cutoff', TRIM_QUALITY, '--compression-level', '1', '-m', str(MIN_LENGTH), '--times', str(ADAPTER_TIMES)]
        ca += ['-g', A_KEY, '-G', A_KEY]
        for s in B_KEYS:
            ca += ['-g', s, '-G', s]
        ca += ['-a', PLATFORM_ADAPTER, '-A', PLATFORM_ADAPTER]
        ca += ['-o', str(o1), '-p', str(o2), str(r1), str(r2)]

        print(f'  [{bn}] cutadapt ...')
        rr = subprocess.run(ca, capture_output=True, text=True)
        if rr.returncode != 0:
            raise RuntimeError(f'cutadapt failed {bn}: {rr.stderr[:800]}')

    n = len(list(out.glob('*.trim.fastq.gz')))
    print(f'[adapter_trim] DONE: {n} trimmed files')


### Run

Run human adapter/MID trimming after reviewing parameters above.

In [ ]:
run_adapter_trim('/data/user/epishkin', 'PRJEB40348')
